In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sqlite3
import pandas as pd

db_path = '/content/drive/Shareddrives/Hotels/Full_HotelRec/HotelRec.db'

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
print(f"Successfully connected to {db_path}")

# Let's see what tables are available in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nTables in the database:")
for table in tables:
    print(table[0])

Successfully connected to /content/drive/Shareddrives/Hotels/Full_HotelRec/HotelRec.db

Tables in the database:
hotel_reviews


In [ ]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [ ]:
# Show schema
schema_df = pd.read_sql_query("PRAGMA table_info(hotel_reviews);", conn)
print(schema_df)

# Show a few rows
sample_df = pd.read_sql_query("SELECT * FROM hotel_reviews LIMIT 5;", conn)
sample_df

    cid           name  type  notnull dflt_value  pk
0     0      hotel_url  TEXT        0       None   0
1     1         author  TEXT        0       None   0
2     2           date  TEXT        0       None   0
3     3         rating  REAL        0       None   0
4     4          title  TEXT        0       None   0
5     5           text  TEXT        0       None   0
6     6  sleep_quality  REAL        0       None   0
7     7          value  REAL        0       None   0
8     8          rooms  REAL        0       None   0
9     9        service  REAL        0       None   0
10   10    cleanliness  REAL        0       None   0
11   11       location  REAL        0       None   0


,hotel_url,author,date,rating,title,text,sleep_quality,value,rooms,service,cleanliness,location
0,Hotel_Review-g194775-d1121769-Reviews-Hotel_Ba...,violettaf340,2019-01-01T00:00:00,5.0,Xmas holiday,We went here with our kids for Xmas holiday an...,NaN,NaN,NaN,NaN,NaN,NaN
1,Hotel_Review-g194775-d1121769-Reviews-Hotel_Ba...,Lagaiuzza,2016-01-01T00:00:00,5.0,"Baltic, what else?",We have spent in this hotel our summer holiday...,NaN,NaN,NaN,NaN,NaN,NaN
2,Hotel_Review-g194775-d1121769-Reviews-Hotel_Ba...,ashleyn763,2014-10-01T00:00:00,5.0,Excellent in every way!,I visited Hotel Baltic with my husband for som...,NaN,5.0,NaN,5.0,NaN,5.0
3,Hotel_Review-g194775-d1121769-Reviews-Hotel_Ba...,DavideMauro,2014-08-01T00:00:00,5.0,The house of your family's holiday,I've travelled quite a numbers of hotels but t...,5.0,NaN,NaN,5.0,5.0,NaN
4,Hotel_Review-g194775-d1121769-Reviews-Hotel_Ba...,Alemma11,2013-08-01T00:00:00,4.0,"A paradise for children (and parents, of course)",We decided for this family holiday destination...,3.0,4.0,4.0,5.0,3.0,4.0


In [ ]:
quick_years = pd.read_sql_query("""
SELECT SUBSTR(date, 1, 4) AS year, COUNT(*) AS n
FROM (
    SELECT date
    FROM hotel_reviews
    LIMIT 5000000
)
GROUP BY year
ORDER BY year;
""", conn)

print(quick_years)

    year       n
0   2001      20
1   2002     509
2   2003    3789
3   2004    9496
4   2005   16907
5   2006   27166
6   2007   41221
7   2008   53743
8   2009   88302
9   2010  125584
10  2011  212820
11  2012  339822
12  2013  472508
13  2014  570072
14  2015  700934
15  2016  802673
16  2017  747828
17  2018  622000
18  2019  164606


In [ ]:
eligible_users = pd.read_sql_query("""
    SELECT DISTINCT author
    FROM hotel_reviews
    WHERE date >= '2016-01-01'
""", conn)

sampled_users = eligible_users.sample(n=100000, random_state=42)

conn.execute("DROP TABLE IF EXISTS sampled_users;")
conn.execute("CREATE TEMP TABLE sampled_users(author TEXT);")

conn.executemany(
    "INSERT INTO sampled_users(author) VALUES (?);",
    [(u,) for u in sampled_users["author"].tolist()]
)
conn.commit()

subset_df = pd.read_sql_query("""
    SELECT *
    FROM hotel_reviews
    WHERE date >= '2016-01-01'
      AND author IN (SELECT author FROM sampled_users);
""", conn)

print("rows:", len(subset_df))
print("users:", subset_df["author"].nunique())
print("hotels:", subset_df["hotel_url"].nunique())
print("avg reviews/user:", subset_df.groupby("author").size().mean())
print("median reviews/user:", subset_df.groupby("author").size().median())
print("avg reviews/hotel:", subset_df.groupby("hotel_url").size().mean())
print("median reviews/hotel:", subset_df.groupby("hotel_url").size().median())

# Applied temporal filter, cutoff taking 2016 onwards
# Chose temporal to improve data relevance and maintain density
# From this filtered subset, randomly sampled users, and kept all interactions (rows) from these users witin 2016-onwards window
# Chose user sampling instead of row sampling to keep full behavior for eaach user intact

rows: 181300
users: 100000
hotels: 88536
avg reviews/user: 1.813
median reviews/user: 1.0
avg reviews/hotel: 2.04775458570525
median reviews/hotel: 1.0


In [ ]:
cols_to_drop = ["text", "title"]
subset_df = subset_df.drop(columns=cols_to_drop)
subset_df.to_csv("hotelrec_subset_2016plus_100kusers.csv", index=False)

# drop unused title and text and save subset csv

In [ ]:
df = pd.read_csv("hotelrec_subset_2016plus_100kusers.csv")

print(df.shape)
print(df.columns)
print(df.dtypes)

# sanity check, examine subset csv

(181300, 10)
Index(['hotel_url', 'author', 'date', 'rating', 'sleep_quality', 'value',
       'rooms', 'service', 'cleanliness', 'location'],
      dtype='object')
hotel_url         object
author            object
date              object
rating           float64
sleep_quality    float64
value            float64
rooms            float64
service          float64
cleanliness      float64
location         float64
dtype: object


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# convert date to datetime and sort

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# add basic time features

In [ ]:
# User prior review count
df["user_review_count_prior"] = df.groupby("author").cumcount()

# User prior average rating
df["user_avg_rating_prior"] = (
    df.groupby("author")["rating"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=0, drop=True)
)

#for each user find how many past reviews they have written before the current one and the average of their prior ratings

In [ ]:
# Hotel prior review count
df["hotel_review_count_prior"] = df.groupby("hotel_url").cumcount()

# Hotel prior average rating
df["hotel_avg_rating_prior"] = (
    df.groupby("hotel_url")["rating"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=0, drop=True)
)

# for each hotel find how many past reviews it has before the current one and the average of its prior ratings

In [ ]:
df[[
    "author", "hotel_url", "date", "rating",
    "user_review_count_prior", "user_avg_rating_prior",
    "hotel_review_count_prior", "hotel_avg_rating_prior"
]].head(15)

# sanity check to ensure no leakage (prevent model from seeing future info which give artifically high performance)

,author,hotel_url,date,rating,user_review_count_prior,user_avg_rating_prior,hotel_review_count_prior,hotel_avg_rating_prior
0,Tina3302,Hotel_Review-g6686791-d620384-Reviews-Sveltos_...,2016-01-01,3.0,0,NaN,0,NaN
1,mamdouhm461,Hotel_Review-g294205-d299629-Reviews-Pavillon_...,2016-01-01,3.0,0,NaN,0,NaN
2,steves892,Hotel_Review-g54370-d1026851-Reviews-Wingate_b...,2016-01-01,5.0,0,NaN,0,NaN
3,M3294AGbarbarah,Hotel_Review-g36052-d90945-Reviews-Best_Wester...,2016-01-01,5.0,0,NaN,0,NaN
4,sandrafbourne,Hotel_Review-g186291-d571831-Reviews-Premier_I...,2016-01-01,5.0,0,NaN,0,NaN
5,picaroonschest,Hotel_Review-g504216-d1526129-Reviews-Ocean_Ho...,2016-01-01,1.0,0,NaN,0,NaN
6,maiboston,Hotel_Review-g33101-d84687-Reviews-Courtyard_S...,2016-01-01,3.0,0,NaN,0,NaN
7,JannePorkkud,Hotel_Review-g295424-d325646-Reviews-Le_Meridi...,2016-01-01,4.0,0,NaN,0,NaN
8,Aaron212,Hotel_Review-g60982-d87102-Reviews-Moana_Surfr...,2016-01-01,5.0,0,NaN,0,NaN
9,233lesleyp,Hotel_Review-g54774-d650648-Reviews-GrandStay_...,2016-01-01,4.0,0,NaN,0,NaN


In [ ]:
df["user_has_history"] = (df["user_review_count_prior"] > 0).astype(int)
df["hotel_has_history"] = (df["hotel_review_count_prior"] > 0).astype(int)

# add binary indicators to explicitly inform xgboost of cold start cases (no prior interactions)

In [ ]:
# add aspect based features for hotels
aspects = ["sleep_quality", "value", "rooms", "service", "cleanliness", "location"]

for col in aspects:
    df[f"hotel_avg_{col}_prior"] = (
        df.groupby("hotel_url")[col]
          .apply(lambda x: x.expanding().mean().shift(1))
          .reset_index(level=0, drop=True)
    )

# for each hotel and each aspect, find average service, cleanliness, etc before this review
# opt to not use user based features because user history is weaker, can decide to add later if needed

In [ ]:
df[[
    "hotel_url", "date",
    "service",
    "hotel_avg_service_prior"
]].head(15)

# sanity check, first occurrence should be nan and later averages build correctly

,hotel_url,date,service,hotel_avg_service_prior
0,Hotel_Review-g6686791-d620384-Reviews-Sveltos_...,2016-01-01,4.0,NaN
1,Hotel_Review-g294205-d299629-Reviews-Pavillon_...,2016-01-01,5.0,NaN
2,Hotel_Review-g54370-d1026851-Reviews-Wingate_b...,2016-01-01,4.0,NaN
3,Hotel_Review-g36052-d90945-Reviews-Best_Wester...,2016-01-01,5.0,NaN
4,Hotel_Review-g186291-d571831-Reviews-Premier_I...,2016-01-01,5.0,NaN
5,Hotel_Review-g504216-d1526129-Reviews-Ocean_Ho...,2016-01-01,NaN,NaN
6,Hotel_Review-g33101-d84687-Reviews-Courtyard_S...,2016-01-01,NaN,NaN
7,Hotel_Review-g295424-d325646-Reviews-Le_Meridi...,2016-01-01,4.0,NaN
8,Hotel_Review-g60982-d87102-Reviews-Moana_Surfr...,2016-01-01,4.0,NaN
9,Hotel_Review-g54774-d650648-Reviews-GrandStay_...,2016-01-01,NaN,NaN


In [ ]:
# 90/5/5 train/validate/test split, preserving temporal ordering
n = len(df)

train_end = int(0.90 * n)
val_end   = int(0.95 * n)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]
# chose 90/5/5 to maximize data available for training given this dataset's extreme sparsity

# check that train < val < test chronologically
print(train["date"].min(), train["date"].max())
print(val["date"].min(), val["date"].max())
print(test["date"].min(), test["date"].max())

2016-01-01 00:00:00 2018-11-01 00:00:00
2018-11-01 00:00:00 2019-02-01 00:00:00
2019-02-01 00:00:00 2019-05-13 06:35:31


In [ ]:
# build feature matrix to use for xgboost training
features = [
    col for col in df.columns
    if col not in ["author", "hotel_url", "date", "rating"]
]
#drop non-feature cols, to model: (user behavior, hotel behavior, context) → rating

In [ ]:
#create datasets
X_train = train[features]
y_train = train["rating"]

X_val = val[features]
y_val = val["rating"]

X_test = test[features]
y_test = test["rating"]

In [ ]:
# train xgboost w/ 200 estimators
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

[0]	validation_0-rmse:1.09617
[1]	validation_0-rmse:1.07190
[2]	validation_0-rmse:1.04312
[3]	validation_0-rmse:1.01976
[4]	validation_0-rmse:0.99984
[5]	validation_0-rmse:0.98322
[6]	validation_0-rmse:0.97256
[7]	validation_0-rmse:0.96268
[8]	validation_0-rmse:0.95473
[9]	validation_0-rmse:0.94820
[10]	validation_0-rmse:0.93912
[11]	validation_0-rmse:0.93442
[12]	validation_0-rmse:0.92758
[13]	validation_0-rmse:0.92280
[14]	validation_0-rmse:0.91827
[15]	validation_0-rmse:0.91430
[16]	validation_0-rmse:0.91204
[17]	validation_0-rmse:0.90906
[18]	validation_0-rmse:0.90651
[19]	validation_0-rmse:0.90526
[20]	validation_0-rmse:0.90332
[21]	validation_0-rmse:0.90195
[22]	validation_0-rmse:0.90038
[23]	validation_0-rmse:0.89968
[24]	validation_0-rmse:0.89868
[25]	validation_0-rmse:0.89806
[26]	validation_0-rmse:0.89737
[27]	validation_0-rmse:0.89670
[28]	validation_0-rmse:0.89620
[29]	validation_0-rmse:0.89574
[30]	validation_0-rmse:0.89530
[31]	validation_0-rmse:0.89493
[32]	validation_0-

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# evaluate w/200 estimators
from sklearn.metrics import mean_squared_error
import numpy as np

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("Test RMSE:", rmse)

Test RMSE: 0.8995025483600807


In [ ]:
# print feature importance for report
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False).head(15))

service                        0.427891
value                          0.146616
rooms                          0.124491
cleanliness                    0.097757
sleep_quality                  0.049801
hotel_avg_rating_prior         0.030286
location                       0.024836
user_avg_rating_prior          0.021382
hotel_review_count_prior       0.007675
hotel_has_history              0.007636
hotel_avg_service_prior        0.006987
user_review_count_prior        0.006819
hotel_avg_cleanliness_prior    0.006425
hotel_avg_location_prior       0.006393
year                           0.006318
dtype: float32


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from itertools import product

param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5]
}

results = []

keys = list(param_grid.keys())
values = list(param_grid.values())

for combo in product(*values):
    params = dict(zip(keys, combo))

    model = XGBRegressor(
        n_estimators=1000,
        early_stopping_rounds=50,
        random_state=42,
        **params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds_val = model.predict(X_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))

    results.append({
        **params,
        "best_iteration": model.best_iteration,
        "val_rmse": rmse_val
    })

results_df = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
print(results_df.head(10))

   max_depth  learning_rate  subsample  colsample_bytree  min_child_weight  \
0          6           0.03        0.8               1.0                 5   
1          6           0.03        0.8               1.0                 1   
2          6           0.03        0.8               0.8                 5   
3          6           0.05        1.0               1.0                 1   
4          6           0.10        0.8               1.0                 5   
5          6           0.05        0.8               1.0                 5   
6          6           0.03        0.8               0.8                 1   
7          6           0.03        1.0               1.0                 5   
8          6           0.05        1.0               1.0                 5   
9          4           0.03        0.8               1.0                 5   

   best_iteration  val_rmse  
0             376  0.889764  
1             275  0.889859  
2             340  0.889935  
3             163  0.

In [ ]:
# train final model using best row from results_df
best_params = results_df.iloc[0].to_dict() #cast parameters to int to fix typing error
best_params.pop("val_rmse")
best_params.pop("best_iteration")

best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])
best_params["learning_rate"] = float(best_params["learning_rate"])
best_params["subsample"] = float(best_params["subsample"])
best_params["colsample_bytree"] = float(best_params["colsample_bytree"])

final_model = XGBRegressor(
    n_estimators=1000,
    early_stopping_rounds=50,
    random_state=42,
    **best_params
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

preds_test = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, preds_test))

print("Best params:", best_params)
print("Test RMSE:", test_rmse)

[0]	validation_0-rmse:1.12267
[1]	validation_0-rmse:1.11139
[2]	validation_0-rmse:1.10066
[3]	validation_0-rmse:1.09053
[4]	validation_0-rmse:1.08082
[5]	validation_0-rmse:1.07162
[6]	validation_0-rmse:1.06284
[7]	validation_0-rmse:1.05452
[8]	validation_0-rmse:1.04663
[9]	validation_0-rmse:1.03906
[10]	validation_0-rmse:1.03199
[11]	validation_0-rmse:1.02515
[12]	validation_0-rmse:1.01862
[13]	validation_0-rmse:1.01243
[14]	validation_0-rmse:1.00662
[15]	validation_0-rmse:1.00107
[16]	validation_0-rmse:0.99589
[17]	validation_0-rmse:0.99086
[18]	validation_0-rmse:0.98614
[19]	validation_0-rmse:0.98160
[20]	validation_0-rmse:0.97733
[21]	validation_0-rmse:0.97336
[22]	validation_0-rmse:0.96953
[23]	validation_0-rmse:0.96584
[24]	validation_0-rmse:0.96233
[25]	validation_0-rmse:0.95902
[26]	validation_0-rmse:0.95585
[27]	validation_0-rmse:0.95296
[28]	validation_0-rmse:0.95014
[29]	validation_0-rmse:0.94752
[30]	validation_0-rmse:0.94494
[31]	validation_0-rmse:0.94249
[32]	validation_0-

===========================

XGBOOST WITH FULL DATASET

===========================

In [ ]:
pd.read_sql_query("PRAGMA index_list('hotel_reviews');", conn) # check if index on date exists

# date index does not exist

,seq,name,unique,origin,partial


In [ ]:
conn.execute("CREATE INDEX idx_date ON hotel_reviews(date);")
conn.commit()

In [ ]:
import time

query = """
SELECT hotel_url, author, date, rating,
       CAST(substr(date, 1, 4) AS INTEGER) AS year,
       CAST(substr(date, 6, 2) AS INTEGER) AS month,
       sleep_quality, value, rooms, service, cleanliness, location
FROM hotel_reviews
ORDER BY date
"""

chunk_iter = pd.read_sql_query(query, conn, chunksize=200_000)

start_time = time.time()
total_rows_processed = 0

In [ ]:
from collections import defaultdict
import pandas as pd

# running stats for overall ratings
user_rating_sum = defaultdict(float)
user_rating_count = defaultdict(int)

hotel_rating_sum = defaultdict(float)
hotel_rating_count = defaultdict(int)

# running stats for hotel aspect averages
aspects = ["sleep_quality", "value", "rooms", "service", "cleanliness", "location"]
hotel_aspect_sum = {a: defaultdict(float) for a in aspects}
hotel_aspect_count = {a: defaultdict(int) for a in aspects}

output_path = "hotelrec_model_ready_full_streamed.csv"
first_write = True

for chunk_num, chunk in enumerate(chunk_iter, start=1):
    chunk_start = time.time()
    chunk_size = len(chunk)
    total_rows_processed += chunk_size

    print(f"\n--- Processing chunk {chunk_num} ---")
    print(f"Chunk size: {chunk_size:,}")
    print(f"Total rows processed so far: {total_rows_processed:,}")

    rows_out = []

    for row in chunk.itertuples(index=False):
        user = row.author
        hotel = row.hotel_url
        rating = row.rating

        # prior user features
        user_count_prior = user_rating_count[user]
        user_avg_rating_prior = (
            user_rating_sum[user] / user_count_prior
            if user_count_prior > 0 else None
        )

        # prior hotel features
        hotel_count_prior = hotel_rating_count[hotel]
        hotel_avg_rating_prior = (
            hotel_rating_sum[hotel] / hotel_count_prior
            if hotel_count_prior > 0 else None
        )

        # build output row
        out = {
            "date": row.date,
            "rating": rating,
            "year": row.year,
            "month": row.month,
            "user_review_count_prior": user_count_prior,
            "user_avg_rating_prior": user_avg_rating_prior,
            "hotel_review_count_prior": hotel_count_prior,
            "hotel_avg_rating_prior": hotel_avg_rating_prior,
            "user_has_history": int(user_count_prior > 0),
            "hotel_has_history": int(hotel_count_prior > 0),
        }

        # prior hotel aspect averages
        for a in aspects:
            cnt = hotel_aspect_count[a][hotel]
            out[f"hotel_avg_{a}_prior"] = (
                hotel_aspect_sum[a][hotel] / cnt if cnt > 0 else None
            )

        rows_out.append(out)

        # update AFTER computing features
        if pd.notna(rating):
            user_rating_sum[user] += rating
            user_rating_count[user] += 1

            hotel_rating_sum[hotel] += rating
            hotel_rating_count[hotel] += 1

        for a in aspects:
            val = getattr(row, a)
            if pd.notna(val):
                hotel_aspect_sum[a][hotel] += val
                hotel_aspect_count[a][hotel] += 1

    out_df = pd.DataFrame(rows_out)

    out_df.to_csv(
        output_path,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )
    first_write = False

    chunk_time = time.time() - chunk_start
    total_time = time.time() - start_time

    print(f"Chunk {chunk_num} done in {chunk_time:.2f} sec")
    print(f"Total elapsed time: {total_time/60:.2f} minutes")
    print(f"Last date in chunk: {chunk['date'].iloc[-1]}")

print("\nFinished streaming feature generation.")
print(f"Output saved to: {output_path}")
print(f"Total rows processed: {total_rows_processed:,}")


--- Processing chunk 1 ---
Chunk size: 200,000
Total rows processed so far: 200,000
Chunk 1 done in 4.75 sec
Total elapsed time: 1.70 minutes
Last date in chunk: 2005-05-01T00:00:00

--- Processing chunk 2 ---
Chunk size: 200,000
Total rows processed so far: 400,000
Chunk 2 done in 5.30 sec
Total elapsed time: 2.17 minutes
Last date in chunk: 2006-05-01T00:00:00

--- Processing chunk 3 ---
Chunk size: 200,000
Total rows processed so far: 600,000
Chunk 3 done in 5.82 sec
Total elapsed time: 2.64 minutes
Last date in chunk: 2007-01-01T00:00:00

--- Processing chunk 4 ---
Chunk size: 200,000
Total rows processed so far: 800,000
Chunk 4 done in 6.06 sec
Total elapsed time: 3.10 minutes
Last date in chunk: 2007-07-01T00:00:00

--- Processing chunk 5 ---
Chunk size: 200,000
Total rows processed so far: 1,000,000
Chunk 5 done in 6.28 sec
Total elapsed time: 3.51 minutes
Last date in chunk: 2008-01-01T00:00:00

--- Processing chunk 6 ---
Chunk size: 200,000
Total rows processed so far: 1,200,

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

model_df = pd.read_csv("hotelrec_model_ready_full_streamed.csv")
print(model_df.shape)
print(model_df.columns)
print(model_df.dtypes)

# load processed csv

(50264364, 16)
Index(['date', 'rating', 'year', 'month', 'user_review_count_prior',
       'user_avg_rating_prior', 'hotel_review_count_prior',
       'hotel_avg_rating_prior', 'user_has_history', 'hotel_has_history',
       'hotel_avg_sleep_quality_prior', 'hotel_avg_value_prior',
       'hotel_avg_rooms_prior', 'hotel_avg_service_prior',
       'hotel_avg_cleanliness_prior', 'hotel_avg_location_prior'],
      dtype='object')
date                              object
rating                           float64
year                               int64
month                              int64
user_review_count_prior            int64
user_avg_rating_prior            float64
hotel_review_count_prior           int64
hotel_avg_rating_prior           float64
user_has_history                   int64
hotel_has_history                  int64
hotel_avg_sleep_quality_prior    float64
hotel_avg_value_prior            float64
hotel_avg_rooms_prior            float64
hotel_avg_service_prior          flo

In [ ]:
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)
# convert date to datetime and sort to double check

In [ ]:
n = len(model_df)
train_end = int(0.95 * n)

train = model_df.iloc[:train_end]
test = model_df.iloc[train_end:]

print(train["date"].min(), train["date"].max())
print(test["date"].min(), test["date"].max())
print(train.shape, test.shape)
# temporal 95/5 train/test split

2001-02-01 00:00:00 2018-11-01 00:00:00
2018-11-01 00:00:00 2019-09-20 00:00:00
(47751145, 16) (2513219, 16)


In [ ]:
features = [c for c in model_df.columns if c not in ["date", "rating"]]

X_train = train[features]
y_train = train["rating"]

X_test = test[features]
y_test = test["rating"]
# build feature matrix

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=1.0,
    min_child_weight=5,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=20
)

n_sample = min(100000, len(X_test))

val_sample = X_test.sample(n=n_sample, random_state=42)
y_val_sample = y_test.loc[val_sample.index]

start = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(val_sample, y_val_sample)],
    verbose=10
)

fit_time = time.time() - start
print("Total fit time (minutes):", fit_time / 60)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("Test RMSE:", rmse)

# train xgboost on first temporal 95% of full dataset using previously obtained best parameters, test rmse from most recent 5%

[0]	validation_0-rmse:1.13714
[10]	validation_0-rmse:1.08975
[20]	validation_0-rmse:1.06190
[30]	validation_0-rmse:1.04557
[40]	validation_0-rmse:1.03592
[50]	validation_0-rmse:1.03015
[60]	validation_0-rmse:1.02660
[70]	validation_0-rmse:1.02432
[80]	validation_0-rmse:1.02287
[90]	validation_0-rmse:1.02186
[100]	validation_0-rmse:1.02124
[110]	validation_0-rmse:1.02073
[120]	validation_0-rmse:1.02027
[130]	validation_0-rmse:1.01997
[140]	validation_0-rmse:1.01967
[150]	validation_0-rmse:1.01941
[160]	validation_0-rmse:1.01920
[170]	validation_0-rmse:1.01902
[180]	validation_0-rmse:1.01889
[190]	validation_0-rmse:1.01874
[200]	validation_0-rmse:1.01865
[210]	validation_0-rmse:1.01853
[220]	validation_0-rmse:1.01842
[230]	validation_0-rmse:1.01833
[240]	validation_0-rmse:1.01823
[250]	validation_0-rmse:1.01814
[260]	validation_0-rmse:1.01807
[270]	validation_0-rmse:1.01802
[280]	validation_0-rmse:1.01793
[290]	validation_0-rmse:1.01787
[299]	validation_0-rmse:1.01782
Total fit time (min